<a href="https://colab.research.google.com/github/Mandiparaut/BDAT1004_FinalProject/blob/main/Datajam_workshop_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
  <img src="https://img.shields.io/badge/DataJam-2026-blue?style=for-the-badge&logo=python&logoColor=white" alt="DataJam 2026"/>
  <img src="https://img.shields.io/badge/CGI-Partner-red?style=for-the-badge" alt="CGI Partner"/>
  <img src="https://img.shields.io/badge/SMU_MBAN-Workshop-green?style=for-the-badge" alt="SMU MBAN"/>
</p>

---

# 🌳 Data Collection & Wrangling Workshop
## Urban Heat & Tree Canopy: Thermal Inequity in Halifax

<p align="center">
  <strong>🌡️ Where should we plant trees to help the people who need it most? 🌡️</strong>
</p>

---

**Workshop Overview:**
| | |
|---|---|
| ⏱️ **Duration** | 2 hours |
| 🎯 **Goal** | Learn to collect data from the web and clean it for analysis |
| 🌡️ **Theme** | Identifying thermal inequity in Halifax using tree, heat, and building data |
| 📊 **Fun Fact** | Data scientists spend ~73% of their time on data preparation! |

---

### Our Question Today:

> **"How can we identify and mitigate thermal inequity in Halifax by strategically expanding the urban canopy?"**

Translation: Some neighborhoods are hotter than others because they have fewer trees. Where should we plant more trees to help the people who need it most?

---

### What You'll Learn:

| Section | Topic | Time |
|---------|-------|------|
| **Part 1** | **Data Collection** | |
| 1 | Open Data Portals (Halifax Trees & Buildings) | 10 min |
| 2 | APIs (iNaturalist) | 10 min |
| 3 | Web Scraping (Halifax Tree Project) | 10 min |
| 4 | Pre-Collected Data (Heat & Building Density) | 5 min |
| **Part 2** | **Data Wrangling with pandas** | |
| 4.1-4.4 | Inspect, Diagnose, Clean, Transform | 35 min |
| 4.4b | **Combining Data: Trees + Heat + Buildings** | 10 min |
| 4.5 | Validate & Save | 5 min |

---

<p align="center">
  <em>Let's dive in!</em> 🚀
</p>

---
# ⚙️ Setup

Before we begin, let's get everything ready!

## Step 1: Copy the Workshop Folder to Your Drive

1. **Open the shared folder link** (provided by instructor):
   - Look for: `HalifaxDatajam2026_Workshop1_Content`

2. **Make a copy in your own Drive:**
   - Right-click on the folder → **"Make a copy"**
   - Or: Click the folder, then **File → Make a copy**
   - This copies both the notebook AND all the backup data files!

3. **Open YOUR copy of this notebook** from your Drive
   - Navigate to your copy of the folder in Google Drive
   - Double-click on the notebook file to open it in Colab

> **Important:** Make sure you're working in YOUR copy, not the shared original!

In [ ]:
# ============================================================
# STEP 1: Test your setup (Run this cell first!)
# ============================================================
# Press the ▶ Play button or hit Shift+Enter

import requests
import pandas as pd
from bs4 import BeautifulSoup

print("✅ All libraries loaded successfully!")
print("✅ You're ready for the workshop!")

In [ ]:
# ============================================================
# STEP 2: Mount Google Drive
# ============================================================
# This connects Colab to your Google Drive so we can access the workshop files.

from google.colab import drive
drive.mount('/content/drive')

# Set the path to your workshop folder
# (This should be your COPY of HalifaxDatajam2026_Workshop1_Content)
workshop_folder = '/content/drive/MyDrive/HalifaxDatajam2026_Workshop1_Content'

# Verify the folder exists and show what's inside
import os
if os.path.exists(workshop_folder):
    print(f"✅ Workshop folder found!")
    print(f"📁 Location: {workshop_folder}")
    print(f"\n📋 Contents:")
    for item in os.listdir(workshop_folder):
        print(f"   - {item}")
else:
    print("❌ Workshop folder not found!")
    print("   Make sure you copied 'HalifaxDatajam2026_Workshop1_Content' to your Drive")
    print("   and are running YOUR copy of the notebook.")

---
# Part 1: Data Collection 📥

> **Important:** In Part 1, we're **only collecting** data - not cleaning it. We'll save cleaning for Part 2!

In [ ]:
# Libraries already imported in Setup section above!
# Just confirming they're loaded:

print("📦 Libraries available:")
print(f"   - requests (for APIs)")
print(f"   - pandas (for data manipulation)")
print(f"   - BeautifulSoup (for web scraping)")
print("\n✅ Ready to collect data!")

---
---
# 🌳 Section 1: Open Data Portals - Halifax Street Trees & Buildings

## The Gold Standard: Structured Government Data

Halifax has an **Open Data Hub** where the city publishes datasets for public use. This is the best place to start when looking for data!

### 🔍 Step 1: Explore the Website

**Before writing any code, let's explore the data source:**

👉 **Open this link in a new tab:** [Halifax Open Data Hub - Public Trees](https://data-hrm.hub.arcgis.com/datasets/HRM::public-trees/about)

Take 2 minutes to explore:
- What information is available about the dataset?
- How many trees are in the inventory?
- What download options do you see?

### 📥 Datasets We'll Download

For our thermal inequity analysis, we need two datasets from the Halifax Open Data Hub:

| Dataset | Why We Need It | Link |
|---------|---------------|------|
| **Public Trees** | Where shade exists (and where it doesn't) | [Download](https://data-hrm.hub.arcgis.com/datasets/HRM::public-trees/about) |
| **Building Footprints** | Where trees *can't* easily grow (built-up areas) | [Download](https://data-hrm.hub.arcgis.com/datasets/HRM::building-footprints/about) |

---

## Option A: Download CSVs Directly (Easiest)

The simplest way to get data is to download it directly from the website:

### Trees Data:
1. Go to the [Public Trees dataset page](https://data-hrm.hub.arcgis.com/datasets/HRM::public-trees/about)
2. Click **"Download"** and select **CSV**
3. Save it as `Public_Trees.csv`

### Building Footprints Data:
1. Go to the [Building Footprints dataset page](https://data-hrm.hub.arcgis.com/datasets/HRM::building-footprints/about)
2. Click **"Download"** and select **CSV**
3. Save it as `Building_Footprints.csv`

### Upload to Google Drive:
4. Upload both files to your workshop folder in Google Drive:
   - Open [Google Drive](https://drive.google.com)
   - Navigate to your copy of `HalifaxDatajam2026_Workshop1_Content`
   - Drag and drop the CSV files into the folder
5. Run the cell below to load them!

> **Tip:** If you're having trouble downloading, the backup data is already in your workshop folder!

---

### 🔍 When you're done... Go Explore!

Before moving on, take a few minutes to explore these other open data sources:

- **[Halifax Open Data Hub](https://data-hrm.hub.arcgis.com/)** - Municipal datasets
- **[Statistics Canada](https://www.statcan.gc.ca/)** - Census and demographic data
- **[Open Canada](https://open.canada.ca/)** - Federal government data
- **[Nova Scotia Open Data](https://data.novascotia.ca/)** - Provincial datasets

💬 **Discussion:** What datasets could you combine with tree data to study thermal inequity?

In [ ]:
# ============================================================
# OPTION A: Load CSVs from Google Drive (if you downloaded them)
# ============================================================
# If you downloaded the CSVs from the Halifax Open Data Hub,
# save them in your workshop folder and run this cell.

# Load Trees data
trees_path = f'{workshop_folder}/Public_Trees.csv'
df = pd.read_csv(trees_path)
print(f"✅ Loaded {len(df)} trees from CSV!")

# Load Building Footprints data
buildings_path = f'{workshop_folder}/Building_Footprints.csv'
try:
    df_buildings = pd.read_csv(buildings_path)
    print(f"✅ Loaded {len(df_buildings)} building footprints from CSV!")
except FileNotFoundError:
    print("ℹ️ Building Footprints not found - that's okay, we'll focus on trees for now")
    df_buildings = None

# Preview the trees data
df.head()

---

## Option B: Programmatic Access via API (OPTIONAL - For Those Who Want to Learn)

⚠️ **This section is optional!** We cover APIs in more depth in Section 2. Skip ahead if you already loaded the CSV above.

The Halifax Open Data Hub uses **ArcGIS**, which provides a REST API we can query directly with Python. This is useful when:
- You want to automate data collection
- You need to refresh data regularly
- You only want a subset of the data

### 📖 API Documentation

You can find field definitions and query options here:
- [ArcGIS REST API for Public Trees](https://services2.arcgis.com/11XBiaBYA9Ep0yNJ/arcgis/rest/services/Public_Trees/FeatureServer/0)

This tells you what columns (fields) are available and their data types.

In [ ]:
# ============================================================
# OPTION B: Query the ArcGIS REST API directly (OPTIONAL)
# ============================================================
# Skip this if you already loaded data using Option A above!

# Halifax Public Trees - ArcGIS REST API
trees_url = "https://services2.arcgis.com/11XBiaBYA9Ep0yNJ/arcgis/rest/services/Public_Trees/FeatureServer/0/query"

params = {
    "where": "1=1",           # SQL WHERE clause - 1=1 means "give me everything"
    "outFields": "*",         # Which columns - * means all of them
    "returnGeometry": "true", # Include lat/lon coordinates
    "outSR": "4326",          # Coordinate system (WGS84)
    "resultRecordCount": 2000, # Limit for demo (full dataset is 79,000!)
    "f": "json"               # Return format
}

response = requests.get(trees_url, params=params)
print(f"Status Code: {response.status_code}")
print(f"200 means success! ✅" if response.status_code == 200 else "Something went wrong ❌")

In [ ]:
# Parse the JSON response (OPTIONAL - only if you ran the API cell above)
data = response.json()

# The data has 'features' - each feature is a tree
print(f"Features retrieved: {len(data['features'])}")

# Let's look at one tree
print("\nFirst tree:")
data['features'][0]

In [ ]:
# Convert to a nice DataFrame (OPTIONAL - only if you ran the API cells above)
# Each feature has 'attributes' (the data) and 'geometry' (coordinates)

features = data['features']
records = [f['attributes'] for f in features]

# Add coordinates from geometry
for i, f in enumerate(features):
    if f.get('geometry'):
        records[i]['LONGITUDE'] = f['geometry']['x']
        records[i]['LATITUDE'] = f['geometry']['y']

df = pd.DataFrame(records)

print(f"✅ Loaded {len(df)} trees!")
print(f"Columns: {list(df.columns)}")
df.head()

### 🎉 That's open data! No scraping, no hacking - just asking nicely.

**Key columns:**
- `SP_COMM` - Common name (Red Maple, Norway Maple, etc.)
- `SP_SCIEN` - Scientific name (Acer rubrum)
- `DBH` - Diameter at Breast Height (how thick the trunk is, in cm)
- `INSTYR` - When it was planted
- `LATITUDE/LONGITUDE` - Where it is

📖 **Full field documentation:** [ArcGIS Field Definitions](https://services2.arcgis.com/11XBiaBYA9Ep0yNJ/arcgis/rest/services/Public_Trees/FeatureServer/0)

> **Note:** We're collecting raw data here - we'll clean it in Part 2!

---
### ✅ Section 1 Complete!
---

---
---
# 🦋 Section 2: APIs - iNaturalist

## What is an API?

**API** stands for **Application Programming Interface**. Think of it as a waiter at a restaurant:
- You (the customer) make a **request** ("I'd like the pasta")
- The waiter takes your request to the kitchen (the server)
- The kitchen prepares your order and the waiter brings back the **response** (your pasta!)

### Why Use APIs Instead of Downloading Files?

| Benefit | Explanation |
|---------|-------------|
| **More up-to-date** | APIs give you live data, not a static snapshot |
| **Query specific subsets** | Only get the data you need (e.g., "just maples in Halifax") |
| **Don't download everything** | Get 50 records instead of 50,000 |
| **Automate updates** | Write code once, run it whenever you need fresh data |

---

## Beyond Government Data: Community Science

The municipal tree inventory only covers trees on public rights-of-way. What about:
- Trees in people's yards?
- Invasive species residents are spotting?
- Pests and diseases?

**iNaturalist** is an app where people photograph wildlife and plants, and the community helps identify them. They have a public API!

---

## How Did I Find the Right API Endpoint?

Here's my process for finding what to query:

1. **Go to the API documentation:** [iNaturalist API Reference](https://www.inaturalist.org/pages/api+reference)
2. **Search for what you need:** I used `Ctrl+F` and searched for "observations"
3. **Found this:** `GET /observations` - "Primary endpoint for retrieving observations"
4. **Read the parameters** to understand what filters are available

The docs tell us:
> *"Primary endpoint for retrieving observations. JSON responses are the most information-rich."*

### 📖 API Documentation

👉 **Explore the docs:** [iNaturalist API Documentation](https://api.inaturalist.org/v1/docs/)

Here you can find:
- Available **endpoints** (different types of data you can request)
- **Query parameters** (filters and options)
- **Response structure** (what the data looks like)

### API Query Parameters We'll Use

| Parameter | Description | Example |
|-----------|-------------|--------|
| `place_id` | Geographic area | 27573 = Halifax County |
| `iconic_taxa` | Type of organism | "Plantae" for plants |
| `quality_grade` | Verification level | "research" = verified |
| `taxon_name` | Search for specific species | "Acer" for maples |

In [ ]:
# ============================================================
# iNaturalist API - Halifax plant observations
# ============================================================
# We're using GET /observations - the main endpoint for retrieving observations
# Follow along! Run this cell and check your output.

inaturalist_url = "https://api.inaturalist.org/v1/observations"

params = {
    "place_id": 27573,        # Halifax County
    "iconic_taxa": "Plantae", # Only plants
    "quality_grade": "research", # Verified IDs only
    "per_page": 50
}

response = requests.get(inaturalist_url, params=params)
data = response.json()

print(f"Status: {response.status_code}")
print(f"Total observations in Halifax: {data['total_results']}")
print(f"Retrieved in this request: {len(data['results'])}")

# ✅ CHECKPOINT: Did you get a status of 200 and some results?

## Extraction vs. Cleaning: What's the Difference?

API responses are often **nested** (data inside data inside data). We need to **extract** the fields we want.

| Step | What It Does | Example |
|------|-------------|----------|
| **Extraction** | Reshape data structure, pull out nested fields | Getting `obs['taxon']['name']` from nested JSON |
| **Cleaning** | Fix data quality issues | Removing duplicates, fixing typos, handling missing values |

**Extraction** is about *structure* - making nested data flat so we can analyze it.

**Cleaning** is about *quality* - fixing errors, inconsistencies, and missing data.

In the cell below, we're doing **extraction** (not cleaning yet!):

In [ ]:
# ============================================================
# Flatten the nested JSON data (EXTRACTION, not cleaning!)
# ============================================================
# The API gives us nested data - we need to extract the fields we want
# and put them into a flat table structure.

records = []
for obs in data['results']:
    record = {
        'observation_id': obs.get('id'),
        'observed_on': obs.get('observed_on'),
        'species_guess': obs.get('species_guess'),
        'place_guess': obs.get('place_guess'),
    }

    # Extract nested taxon info (this is EXTRACTION - pulling from nested structure)
    if obs.get('taxon'):
        record['scientific_name'] = obs['taxon'].get('name')
        record['common_name'] = obs['taxon'].get('preferred_common_name')

    # Extract coordinates from nested geojson
    if obs.get('geojson') and obs['geojson'].get('coordinates'):
        record['longitude'] = obs['geojson']['coordinates'][0]
        record['latitude'] = obs['geojson']['coordinates'][1]

    records.append(record)

df_inat = pd.DataFrame(records)
print(f"✅ Created DataFrame with {len(df_inat)} observations")
df_inat.head()

# ✅ CHECKPOINT: You should see a table with columns like observation_id, observed_on, etc.

In [ ]:
# ============================================================
# 🎯 YOUR TURN: Search for a specific type of plant!
# ============================================================
#
# Modify the API query to search specifically for MAPLES.
#
# HINTS:
# 1. Add the parameter "taxon_name" with the genus name for maples
# 2. The genus name for maples is "Acer" (Latin for maple)
# 3. Check the API docs for more parameters:
#    https://api.inaturalist.org/v1/docs/#!/Observations/get_observations
#
# BONUS: Try searching for other plants!
#   - "Quercus" = Oaks
#   - "Picea" = Spruces
#   - "Betula" = Birches
# ============================================================

params = {
    "place_id": 27573,
    "iconic_taxa": "Plantae",
    "quality_grade": "research",
    "taxon_name": "<GENUS_NAME>",  # <-- REPLACE THIS with "Acer"!
    "per_page": 50
}

response = requests.get(inaturalist_url, params=params)
maple_data = response.json()

print(f"Maple observations found: {maple_data['total_results']}")

### Key Insight:

> "APIs give you structured data, but it's often nested. Part of data wrangling is **extracting** (flattening) these structures into something you can analyze. **Cleaning** comes later!"

---
### ✅ Section 2 Complete!
---

---
---
# 🍜 Section 3: Web Scraping with BeautifulSoup

## When There's No API: Scrape It (Responsibly)

The **Halifax Tree Project** is a local organization with expert blog posts about street trees - which species survive here, lessons learned from planting, qualitative context that doesn't exist in any database.
Link: https://www.halifaxtreeproject.com/street-tree-blog

But there's no "Download All" button. This is where **web scraping** comes in.

### What is BeautifulSoup?

**BeautifulSoup** is a Python library that makes it easy to extract data from HTML (web page code). It lets you:
- Search for specific HTML tags (like `<a>` for links, `<p>` for paragraphs)
- Extract text content from those tags
- Navigate the "tree" structure of HTML documents

Think of it like using "Find" in a Word document, but for web pages!

### Ethics First! ⚖️

Before scraping ANY website:
1. ✅ Check `robots.txt` (e.g., [example.com/robots.txt](https://www.halifaxtreeproject.com/robots.txt))
2. ✅ Read the Terms of Service
3. ✅ Add delays between requests (be nice to servers!)
4. ✅ Never scrape personal/private information
5. ✅ Always check if they have an API instead!

---

### ⚠️ IMPORTANT: Run Scraping Code Only Once!

**Be respectful to servers.** Running scraping code repeatedly:
- Puts unnecessary load on the website
- May get your IP address blocked
- Is inconsiderate to the site owners

If you need to re-run, add a delay: `import time; time.sleep(2)`

In [ ]:
# ============================================================
# Fetch the Halifax Tree Project blog page
# ============================================================
# ⚠️ Only run this cell ONCE during the workshop!

import time

url = "https://www.halifaxtreeproject.com/street-tree-blog"
headers = {'User-Agent': 'Mozilla/5.0 (Educational Workshop)'}  # Identify ourselves

response = requests.get(url, headers=headers)
print(f"Status Code: {response.status_code}")

# Let's peek at the raw HTML (first 1000 characters)
print("\nRaw HTML preview:")
print(response.text[:1000])

# Be nice - wait before any additional requests
time.sleep(1)

In [ ]:
# ============================================================
# Parse the HTML with BeautifulSoup
# ============================================================
# BeautifulSoup turns raw HTML text into a searchable "tree" structure
# We can then find specific elements by their tag name, class, or ID

soup = BeautifulSoup(response.text, 'html.parser')
print(soup.prettify()[:2000])

In [ ]:
# Now we can search for elements!
title = soup.find('title')
print(f"Page title: {title.text}")

# Count different elements on the page
print(f"\nPage structure:")
print(f"  - Links (<a> tags): {len(soup.find_all('a'))}")
print(f"  - Paragraphs (<p> tags): {len(soup.find_all('p'))}")
print(f"  - Images (<img> tags): {len(soup.find_all('img'))}")

## The Inspect Workflow 🔍

**This is the most important skill in web scraping!**

### 🎯 YOUR TURN: Open the page in your browser and use Inspect!

1. Open **[Halifax Tree Project Street Tree Blog](https://www.halifaxtreeproject.com/street-tree-blog)** in a new browser tab
2. Right-click on a blog post title and select **"Inspect"** or **"Inspect Element"**
3. Look at the HTML structure in the Developer Tools panel

Search the soup

**Answer these questions:**
- What HTML tag contains the blog post titles?
- What tag contains the links?
- Do they have any class names?

In [ ]:
# ============================================================
# Find all links on the page
# ============================================================
# This is basic link extraction - finding all <a> tags with href attributes

links = soup.find_all('a', href=True)

print(f"Total links found: {len(links)}")

# Filter for content links (longer text, not just nav)
content_links = []
for link in links:
    text = link.get_text(strip=True)
    href = link.get('href', '')
    # Skip empty links and short navigation text
    if text and len(text) > 10 and href:
        content_links.append({'title': text[:80], 'url': href})

print(f"\nContent links found: {len(content_links)}")
print("\nSample links:")
for item in content_links[:5]:
    print(f"  - {item['title'][:50]}...")
    print(f"    URL: {item['url'][:60]}..." if len(item['url']) > 60 else f"    URL: {item['url']}")

In [ ]:
# ============================================================
# BONUS: Extract content from individual tree pages
# ============================================================
#
# THE THINKING/PSEUDOCODE:
# 1. We have a list of links from the main page
# 2. Each link goes to an individual blog post about a tree species
# 3. We want to: visit each link → extract the text content → store it
# 4. We'll use a loop to do this for multiple pages
# 5. We'll add delays between requests to be respectful
#
# ⚠️ Only run this cell ONCE - we're making multiple requests!

import time

# Filter for blog post links (they often have specific URL patterns)
blog_posts = [
    link for link in content_links
    if 'halifaxtreeproject.com' in link.get('url', '')
][:3]  # Limit to 3 pages for demo - be respectful!

print(f"Found {len(blog_posts)} potential blog post links")

# Extract content from each page
tree_content = []

for post in blog_posts:
    url = post['url']
    # Make sure URL is complete
    if not url.startswith('http'):
        url = 'https://www.halifaxtreeproject.com' + url

    try:
        print(f"\nFetching: {post['title'][:40]}...")
        page_response = requests.get(url, headers=headers, timeout=10)

        if page_response.status_code == 200:
            page_soup = BeautifulSoup(page_response.text, 'html.parser')

            # Extract all paragraph text
            paragraphs = page_soup.find_all('p')
            content_text = ' '.join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])

            if content_text:
                tree_content.append({
                    'title': post['title'],
                    'url': url,
                    'excerpt': content_text[:300] + '...' if len(content_text) > 300 else content_text
                })
                print(f"  ✅ Extracted {len(content_text)} characters")
            else:
                print(f"  ⚠️ No paragraph content found (might be JavaScript-loaded)")
        else:
            print(f"  ❌ Status {page_response.status_code}")

        time.sleep(2)  # Be respectful - wait 2 seconds between requests

    except Exception as e:
        print(f"  ❌ Error: {e}")

print(f"\n" + "="*50)
print(f"""✅ Extracted content from {len(tree_content)} pages.
(⚠️ Intentional, do not rerun)""")

# Display what we found
for item in tree_content:
    print(f"\n📄 {item['title']}")
    print(f"   {item['excerpt'][:150]}...")

## ⚠️ The JavaScript Problem

**Notice something?** The Halifax Tree Project site is built with **Wix**, which loads content dynamically using JavaScript.

When we use `requests.get()`, we only get the initial HTML - not the content that JavaScript adds later. That's why we might not see all the blog posts!

### The Scraping Thought Process

When scraping a site, here's the thinking:

```
1. GOAL: What content do I want? (blog post titles and descriptions)
2. INSPECT: What HTML elements contain that content? (use browser DevTools)
3. EXTRACT: Write code to find those elements and pull out the text
4. PROBLEM: Is content loaded by JavaScript? (requests won't see it!)
5. SOLUTION: Either find a hidden API, or use Selenium to run JavaScript
```

---

## 🚀 BONUS: Handling JavaScript Sites with Selenium

For sites that load content with JavaScript, `requests` alone won't work. We need to use **Selenium** (or something like it), which controls a real web browser!

### How Selenium Works:

```
requests.get() → Just downloads HTML (no JavaScript execution)
Selenium → Opens a real Chrome browser → Runs JavaScript → Gets fully loaded page
```

### When to Use Selenium:
- Content loads after page load ("lazy loading")
- You need to click buttons or fill forms
- Site requires login
- Content is inside JavaScript frameworks (React, Vue, Angular)

### Example Code (Don't run in workshop - just for reference):

```python
# Install: !pip install selenium webdriver-manager

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

# Set up Chrome options (headless = no visible browser window)
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run without opening browser window
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Create the browser
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

# Navigate to page
driver.get("https://www.halifaxtreeproject.com/street-tree-blog")

# Wait for JavaScript to load (important!)
time.sleep(3)

# Now get the page source (with JavaScript content!)
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

# Extract what you need...
# ...

# Always close the browser when done!
driver.quit()
```

### Pro Tip: Check for Hidden APIs!

Before using Selenium, check the **Network tab** in DevTools. Many JavaScript sites actually fetch data from hidden APIs that you could query directly!

1. Open DevTools (F12)
2. Go to Network tab
3. Filter by "XHR" or "Fetch"
4. Refresh the page
5. Look for API calls returning JSON data

---
### ✅ Section 3 Complete!
---

---
---
# 🌡️ Section 4: Pre-Collected Data - Heat & Building Density

## Why I Pre-Collected This Data For You

Not all data collection is practical for a workshop! These datasets require:
- **Large file downloads** (1.5+ GB compressed)
- **Complex extraction** (parsing thousands of files)
- **Specialized access** (satellite imagery processing)

Rather than have everyone download gigabytes of data and spend time on complex file parsing, I extracted what we need ahead of time so we can focus on the analysis.

---

### 1. Urban Heat Island Intensity (UHII)

**What is Urban Heat Island Effect?**

Urban areas are typically **warmer than surrounding rural areas** because:
- 🏗️ Buildings and pavement absorb heat
- 🚗 Vehicles and AC units release heat
- 🌳 Less vegetation = less cooling from evapotranspiration

**UHI Intensity** measures how many degrees Celsius warmer the urban core is compared to surrounding rural areas.

**Data Source:** [Global Urban Heat Island Intensity Dataset](https://doi.org/10.6084/m9.figshare.24821538) on Figshare
- 1.5 GB compressed archive with 5,372 CSV files
- 20 years of satellite measurements (2001-2021)
- Halifax is identified as **UrbanId 710**

### 2. Building Density (NDBI)

**What is NDBI?**

The **Normalized Difference Built-Up Index** is a satellite-derived measure that indicates how "built-up" an area is:
- **Higher NDBI** = more buildings, pavement, concrete
- **Lower NDBI** = more vegetation, water, open space

This is our proxy for building footprints - areas with high NDBI are where trees CAN'T grow!

---

## The Pre-Collected Datasets

| File | Description |
|------|-------------|
| `halifax_uhii_yearly.csv` | 20 years of satellite-derived UHI measurements |
| `heat_vulnerability_da.csv` | Heat vulnerability + building density by census area |

In [ ]:
# ============================================================
# Load Heat Vulnerability by Census Dissemination Area
# ============================================================
# This dataset combines satellite heat measurements with building density (NDBI)

try:
    heat_vuln_df = pd.read_csv(f'{workshop_folder}/heat_vulnerability_da.csv')
    print(f"✅ Loaded heat vulnerability data: {len(heat_vuln_df)} Dissemination Areas")
    print(f"\n📊 Vulnerability Categories:")
    print(heat_vuln_df['VULNERABILITY_CATEGORY'].value_counts())
    print(f"\n🔑 Key columns:")
    print(f"   - DA_ID: Census Dissemination Area identifier")
    print(f"   - LST_CELSIUS: Land Surface Temperature")
    print(f"   - NDBI: Building density index (higher = more built-up)")
    print(f"   - CANOPY_PCT: Tree canopy coverage percentage")
    print(f"   - HEAT_VULNERABILITY: Combined vulnerability score")
    display(heat_vuln_df.head())
except FileNotFoundError:
    print("⚠️ Couldn't find heat vulnerability data in your workshop folder")
    print(f"   Expected: {workshop_folder}/heat_vulnerability_da.csv")
    heat_vuln_df = None

---
---
# 💾 Save Your Collected Data

Before we move on to Part 2, let's save all the data we collected to Google Drive!

This way:
- You won't lose your work if the session disconnects
- You can reload it later without re-running all the API calls
- You have a backup for Part 2

In [ ]:
# ============================================================
# 💾 Save all collected data to your workshop folder
# ============================================================

# 1. Save Halifax Trees (main dataset for Part 2)
if 'df' in dir() and len(df) > 0:
    df.to_csv(f'{workshop_folder}/halifax_trees_raw.csv', index=False)
    print(f"✅ Saved Halifax Trees: {len(df)} rows")
else:
    print("⚠️ No Halifax Trees data to save (df not found)")

# 2. Save iNaturalist observations
if 'df_inat' in dir() and len(df_inat) > 0:
    df_inat.to_csv(f'{workshop_folder}/inaturalist_observations.csv', index=False)
    print(f"✅ Saved iNaturalist data: {len(df_inat)} observations")
else:
    print("⚠️ No iNaturalist data to save (df_inat not found)")

# 3. Save scraped tree content (if any)
if 'tree_content' in dir() and len(tree_content) > 0:
    df_scraped = pd.DataFrame(tree_content)
    df_scraped.to_csv(f'{workshop_folder}/scraped_tree_content.csv', index=False)
    print(f"✅ Saved scraped content: {len(tree_content)} pages")
else:
    print("ℹ️ No scraped content to save (tree_content not found or empty)")

print(f"\n📁 All files saved to: {workshop_folder}")
print("\n🎉 Data collection complete! Ready for Part 2.")

---
---
# 📥 Backup Option: Pre-Collected Data

**Had trouble with any of the collection methods above?** No problem!

Your workshop folder already contains backup data files. Just run the cell below to load them!

This is also useful if:
- The API is down or slow
- You want to skip ahead to wrangling
- You're reviewing the workshop later offline

In [ ]:
# ============================================================
# 📥 BACKUP OPTION: Load pre-collected data from your workshop folder
# ============================================================
# Your workshop folder already contains backup data files!
# Just run this cell to load the trees data.

backup_path = f'{workshop_folder}/halifax_trees_backup.csv'

try:
    df = pd.read_csv(backup_path)
    print(f"✅ Loaded {len(df)} Halifax trees from backup - ready for wrangling!")
    print(f"👉 Continue to 'Part 2: Data Wrangling with pandas'")
    display(df.head())
except FileNotFoundError:
    print(f"❌ Backup file not found: {backup_path}")
    print("   Make sure you copied the workshop folder to your Drive.")

---
---
# ☕ BREAK TIME (5 minutes)

Stretch, grab a coffee, and get ready for the main event: **Data Wrangling!**

When we come back, we'll clean and analyze that Halifax tree data.

---

---
# Part 2: Data Wrangling with pandas 🐼

We have real Halifax tree data! But is it ready for analysis?

**Spoiler:** No. It never is. 😅

### The Wrangling Pipeline:

```
📥 LOAD → 🔍 INSPECT → 🩺 DIAGNOSE → 🧹 CLEAN → 🔄 TRANSFORM → ✅ VALIDATE
```

## 4.1 Inspect 🔍

You should already have `df` loaded from Part 1 (or from the backup option).

Let's understand what we're working with.

In [ ]:
# Confirm your data is loaded
print(f"✅ Data loaded: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# First look at the data
df.head(10)

In [ ]:
# Check data types and non-null counts
# THIS IS YOUR MOST IMPORTANT DIAGNOSTIC TOOL!
df.info()

In [ ]:
# Statistical summary for numeric columns
df.describe()

In [ ]:
# Check unique values in key columns
print("=" * 50)
print("TOP 10 TREE SPECIES")
print("=" * 50)
print(df['SP_COMM'].value_counts().head(10))

print("\n" + "=" * 50)
print("PLANTING YEARS")
print("=" * 50)
print(f"Range: {df['INSTYR'].min()} to {df['INSTYR'].max()}")
print(f"Most common: {df['INSTYR'].mode().values[0] if len(df['INSTYR'].mode()) > 0 else 'N/A'}")

---
## 🎯 Your Turn: Inspection Practice

Choose your level and explore the data!

| Level | Challenge | Skills Practiced |
|-------|-----------|------------------|
| **1** | Find which years had the most trees planted | `value_counts()`, sorting |
| **2** | Identify the top 5 largest trees by diameter | Filtering, sorting, selecting columns |
| **3** | Create a summary showing species count AND average DBH per species | `groupby()`, `agg()`, multiple statistics |

In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 1 (Beginner)
# ============================================================
# Find the top 10 years when the most trees were planted
# Hint: Use value_counts() on the INSTYR column

# YOUR CODE HERE:


In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 2 (Intermediate)
# ============================================================
# Find the 5 largest trees (by DBH) and show their species and diameter
# Hint: Use nlargest() or sort_values() then select specific columns

# YOUR CODE HERE:


In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 3 (Advanced)
# ============================================================
# Create a summary table showing for each species:
#   - Number of trees
#   - Average DBH
#   - Oldest planting year
# Sort by tree count descending, show top 10

# YOUR CODE HERE:


## 4.2 Diagnose Data Quality Issues 🩺

Let's find the problems before we try to fix them!

In [ ]:
# Check for missing values
print("MISSING VALUES")
print("=" * 40)

missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)

missing_report = pd.DataFrame({
    'Missing': missing,
    'Percent': missing_pct
})

# Only show columns with missing values
print(missing_report[missing_report['Missing'] > 0])

In [ ]:
# Check for duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate Tree IDs: {df['TREEID'].duplicated().sum()}")

In [ ]:
# Check for outliers in DBH (tree diameter)
print("DBH (Tree Diameter) Analysis:")
print(f"  Min: {df['DBH'].min()} cm")
print(f"  Max: {df['DBH'].max()} cm")
print(f"  Mean: {df['DBH'].mean():.1f} cm")

# Anything over 200cm is suspicious (that's a 6+ foot diameter!)
outliers = df[df['DBH'] > 200]
print(f"\n⚠️ Trees with DBH > 200cm: {len(outliers)}")
if len(outliers) > 0:
    print(outliers[['TREEID', 'SP_COMM', 'DBH']].head())

In [ ]:
# Check for inconsistencies in species names
print("Checking for 'Maple' name variants:")
maple_variants = df[df['SP_COMM'].str.contains('Maple', case=False, na=False)]['SP_COMM'].unique()
for v in maple_variants:
    count = len(df[df['SP_COMM'] == v])
    print(f"  '{v}': {count} trees")

### 📋 Data Quality Summary

| Issue | Column | Problem | Solution |
|-------|--------|---------|----------|
| 1 | `DBH` | Outliers >200cm | Cap at 200 or investigate |
| 2 | `INSTYR` | Missing values | Fill with median or flag |
| 3 | `SP_COMM` | Possible inconsistencies | Standardize case |
| 4 | Timestamps | Unix milliseconds | Convert to datetime |

---
## 🎯 Your Turn: Diagnosis Practice

Time to be a data detective! Find issues before they become problems.

| Level | Challenge | Skills Practiced |
|-------|-----------|------------------|
| **1** | Count how many trees are missing coordinates (LATITUDE or LONGITUDE) | `isnull()`, logical operators |
| **2** | Find trees with suspicious planting years (before 1800 or after 2025) | Boolean filtering, range checks |
| **3** | Create a complete data quality report showing missing % for ALL columns | `isnull()`, percentage calculations, formatting |

In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 1 (Beginner)
# ============================================================
# Count how many trees are missing LATITUDE or LONGITUDE
# Hint: Use isnull() and the | (or) operator

# YOUR CODE HERE:


In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 2 (Intermediate)
# ============================================================
# Find trees with suspicious INSTYR values:
#   - Planted before 1800 (too old to be accurate)
#   - Planted after 2025 (in the future!)
# Show the count and a sample of these records

# YOUR CODE HERE:


In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 3 (Advanced)
# ============================================================
# Create a comprehensive data quality report that shows:
#   - Column name
#   - Data type
#   - Number of missing values
#   - Percentage missing
# Sort by percentage missing (descending)

# YOUR CODE HERE:


## 4.3 Clean the Data 🧹

Now let's fix these issues systematically.

**GOLDEN RULE:** Always make a copy before cleaning!

In [ ]:
# ALWAYS start with a copy!
df_clean = df.copy()

print(f"Starting shape: {df_clean.shape}")

In [ ]:
# ISSUE 1: Remove duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['TREEID'])
after = len(df_clean)

print(f"✅ Removed {before - after} duplicate trees")
print(f"Rows remaining: {after}")

In [ ]:
# ISSUE 2: Handle DBH outliers
# Cap at 200cm (reasonable max for street trees)

outlier_count = (df_clean['DBH'] > 200).sum()
df_clean['DBH'] = df_clean['DBH'].clip(upper=200)

print(f"✅ Capped {outlier_count} DBH outliers at 200cm")
print(f"New DBH max: {df_clean['DBH'].max()} cm")

In [ ]:
# ISSUE 3: Standardize species names
# Strip whitespace and convert to Title Case

df_clean['SP_COMM'] = df_clean['SP_COMM'].str.strip().str.title()

print(f"✅ Standardized species names")
print(f"Unique species: {df_clean['SP_COMM'].nunique()}")

In [ ]:
# ISSUE 4: Fill missing INSTYR with median

missing_before = df_clean['INSTYR'].isnull().sum()
median_year = df_clean['INSTYR'].median()
df_clean['INSTYR'] = df_clean['INSTYR'].fillna(median_year)

print(f"✅ Filled {missing_before} missing installation years with median ({median_year:.0f})")

In [ ]:
# ISSUE 5: Convert timestamp columns to datetime (if present)

timestamp_cols = ['ADDDATE', 'MODDATE', 'SDATE']
for col in timestamp_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(df_clean[col], unit='ms', errors='coerce')
        print(f"✅ Converted {col} to datetime")

In [ ]:
# Verify our cleaning worked!
print("📊 CLEANED DATA SUMMARY")
print("=" * 50)
print(f"Shape: {df_clean.shape}")
print(f"\nMissing values in key columns:")
print(f"  DBH: {df_clean['DBH'].isnull().sum()}")
print(f"  INSTYR: {df_clean['INSTYR'].isnull().sum()}")
print(f"  SP_COMM: {df_clean['SP_COMM'].isnull().sum()}")
print(f"\nDBH range: {df_clean['DBH'].min()} to {df_clean['DBH'].max()} cm")

---
## 🎯 Your Turn: Cleaning Practice

Apply what you've learned to clean even more of the data!

| Level | Challenge | Skills Practiced |
|-------|-----------|------------------|
| **1** | Filter out trees with missing species names (SP_COMM is null) | Boolean filtering, `dropna()` |
| **2** | Create a column flagging rows that had ANY missing value before cleaning | `any()`, creating flag columns |
| **3** | Write a reusable function that applies all cleaning steps to any tree DataFrame | Functions, method chaining |

In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 1 (Beginner)
# ============================================================
# Remove any trees where SP_COMM (species name) is missing
# Hint: Use dropna() with the subset parameter, or boolean filtering

# YOUR CODE HERE:


In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 2 (Intermediate)
# ============================================================
# Add a column 'HAD_MISSING' that is True if the original row
# had ANY missing values (before we filled them in)
# Hint: Use df.isnull().any(axis=1) on the ORIGINAL df

# YOUR CODE HERE:


In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 3 (Advanced)
# ============================================================
# Create a reusable function clean_tree_data() that takes a raw
# DataFrame and returns a cleaned version with all our fixes applied
#
# The function should:
#   1. Make a copy of the input
#   2. Remove duplicates by TREEID
#   3. Cap DBH at 200
#   4. Standardize SP_COMM to Title Case
#   5. Fill missing INSTYR with median
#   6. Return the cleaned DataFrame

# YOUR CODE HERE:


## 4.4 Transform & Analyze 🔄

Now for the fun part - let's create useful features and discover stories in this data!

In [ ]:
# FEATURE 1: Estimate tree age from DBH
# Trees grow roughly 1-2 cm per year in diameter

df_clean['EST_AGE_YEARS'] = df_clean['DBH'] / 1.5

print("Estimated tree ages:")
print(df_clean['EST_AGE_YEARS'].describe().round(1))

In [ ]:
# FEATURE 2: Categorize life stages

def life_stage(age):
    if pd.isna(age): return 'Unknown'
    if age < 10: return 'Young'
    elif age < 30: return 'Maturing'
    elif age < 60: return 'Mature'
    else: return 'Legacy'

df_clean['LIFE_STAGE'] = df_clean['EST_AGE_YEARS'].apply(life_stage)

print("Tree life stages:")
print(df_clean['LIFE_STAGE'].value_counts())

In [ ]:
# FEATURE 3: Estimate canopy coverage
# Crown spread is roughly 2.5x DBH

df_clean['CANOPY_RADIUS_M'] = (df_clean['DBH'] * 2.5) / 100
df_clean['CANOPY_AREA_M2'] = 3.14159 * (df_clean['CANOPY_RADIUS_M'] ** 2)

print(f"Total canopy coverage: {df_clean['CANOPY_AREA_M2'].sum():,.0f} square meters")
print(f"Average canopy per tree: {df_clean['CANOPY_AREA_M2'].mean():.1f} m²")

In [ ]:
# ANALYSIS: Check the 10-20-30 species diversity rule
# No species should exceed 10% of trees (disease resilience)

species_pct = (df_clean['SP_COMM'].value_counts() / len(df_clean) * 100).round(1)

print("🌳 SPECIES DIVERSITY CHECK (10% Rule)")
print("=" * 50)
print("\nSpecies exceeding 10%:")
violations = species_pct[species_pct > 10]
if len(violations) > 0:
    for species, pct in violations.items():
        print(f"  ⚠️ {species}: {pct}%")
    print("\n💡 High maple concentration = vulnerability to pests/disease!")
else:
    print("  ✅ All species below 10% - good diversity!")

---
## 🎯 Your Turn: Transform Practice

Create new features and analyze the data!

| Level | Challenge | Skills Practiced |
|-------|-----------|------------------|
| **1** | Create a SHADE_POTENTIAL category based on canopy area | `apply()`, conditional logic |
| **2** | Create a DECADE_PLANTED column (1990s, 2000s, etc.) | Integer division, string formatting |
| **3** | Calculate which species have the fastest growth rate (DBH per year since planting) | Derived metrics, `groupby()` |

In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 1 (Beginner)
# ============================================================
# Create a column called 'SHADE_POTENTIAL':
#   - 'High' if CANOPY_AREA_M2 > 50
#   - 'Medium' if between 20 and 50
#   - 'Low' if under 20
#
# Hint: Use a function similar to life_stage above

def shade_potential(area):
    # YOUR CODE HERE
    pass

# Uncomment to test:
# df_clean['SHADE_POTENTIAL'] = df_clean['CANOPY_AREA_M2'].apply(shade_potential)
# print(df_clean['SHADE_POTENTIAL'].value_counts())

In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 2 (Intermediate)
# ============================================================
# Create a column DECADE_PLANTED that shows which decade each tree
# was planted in (e.g., "1990s", "2000s", "2010s")
# Hint: Use integer division (//) by 10, then multiply by 10

# YOUR CODE HERE:


In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 3 (Advanced)
# ============================================================
# Calculate which species grow fastest!
#
# Steps:
#   1. Calculate estimated age for each tree (we did this: DBH / 1.5)
#   2. Calculate growth rate as DBH / age (cm per year)
#   3. Group by species and calculate average growth rate
#   4. Find top 5 fastest-growing species
#
# Hint: Be careful with division by zero!

# YOUR CODE HERE:


---
## 🔗 4.4b Combining Our Data Sources: Trees, Heat, and Buildings

Now for the powerful part - **combining our datasets** to answer the real question:

> "Where should we plant trees to help the people who need it most?"

We have three key datasets:
- **Tree data** with exact locations (lat/long) and species
- **Heat vulnerability data** by census area (temperature, building density, existing canopy)
- **Building footprints** showing where development exists (where trees CAN'T grow)

By joining these, we can find:
- Which hot neighborhoods have the fewest trees
- How building density relates to heat and tree coverage
- Priority areas for new tree planting (hot + low canopy + space available)

In [ ]:
# ============================================================
# Step 1: Load our datasets (if not already loaded)
# ============================================================
import numpy as np

# Load heat vulnerability data
if 'heat_vuln_df' not in dir() or heat_vuln_df is None:
    heat_vuln_df = pd.read_csv(f'{workshop_folder}/heat_vulnerability_da.csv')
    print(f"✅ Loaded heat vulnerability data: {len(heat_vuln_df)} census areas")

# Load building footprints (if available)
try:
    if 'df_buildings' not in dir() or df_buildings is None:
        df_buildings = pd.read_csv(f'{workshop_folder}/Building_Footprints.csv')
        print(f"✅ Loaded building footprints: {len(df_buildings)} buildings")
except FileNotFoundError:
    print("ℹ️ Building footprints not available - we'll use NDBI as proxy for building density")
    df_buildings = None

print(f"\n📊 Datasets ready for joining:")
print(f"   - Trees: {len(df_clean)} records")
print(f"   - Heat vulnerability: {len(heat_vuln_df)} census areas")
if df_buildings is not None:
    print(f"   - Buildings: {len(df_buildings)} footprints")

### Spatial Join: Assigning Trees to Census Areas

Each tree has coordinates. Each census area (Dissemination Area) has a centroid. We'll assign each tree to its **nearest** census area.

This is a simplified spatial join - in real GIS work you'd use polygon boundaries, but distance-based matching works well for this analysis.

In [ ]:
# ============================================================
# Step 3: Merge tree data with heat vulnerability metrics
# ============================================================

# Join the heat metrics onto our tree data
df_with_heat = df_clean.merge(
    heat_vuln_df[['DA_ID', 'LST_CELSIUS', 'NDBI', 'CANOPY_PCT',
                  'HEAT_VULNERABILITY', 'VULNERABILITY_CATEGORY', 'COMMUNITY',
                  'POPULATION', 'AREA_KM2']],
    on='DA_ID',
    how='left'
)

print(f"✅ Combined dataset: {len(df_with_heat)} trees with heat + building data")
print(f"\n📊 Trees by vulnerability category:")
print(df_with_heat['VULNERABILITY_CATEGORY'].value_counts())

print(f"\n🔑 New columns available:")
print(f"   - LST_CELSIUS: Land Surface Temperature")
print(f"   - NDBI: Building density (higher = more built-up, less space for trees)")
print(f"   - CANOPY_PCT: Existing tree canopy coverage")
print(f"   - HEAT_VULNERABILITY: Combined vulnerability score (0-100)")

df_with_heat.head()

### 🔍 Analysis: Discovering Insights from Combined Data

Now we can ask powerful questions that require BOTH datasets!

In [ ]:
# ============================================================
# INSIGHT 1: Trees vs Heat - Do vulnerable areas have fewer trees?
# ============================================================

trees_by_vuln = df_with_heat.groupby('VULNERABILITY_CATEGORY').agg({
    'TREEID': 'count',
    'DBH': 'mean',
    'CANOPY_AREA_M2': 'sum',
    'LST_CELSIUS': 'mean',
    'NDBI': 'mean'  # Building density
}).round(2)

trees_by_vuln.columns = ['Tree Count', 'Avg DBH (cm)', 'Total Canopy (m²)',
                          'Avg Temp (°C)', 'Avg Building Density']

print("🌡️ TREES BY HEAT VULNERABILITY CATEGORY")
print("=" * 70)
print(trees_by_vuln)

print("\n💡 Key questions to consider:")
print("   - Do 'Critical' areas have fewer trees than 'Low' vulnerability areas?")
print("   - Is there a relationship between building density and tree count?")
print("   - Where are trees most needed?")

In [ ]:
# ============================================================
# INSIGHT 2: The Building-Heat-Tree Triangle
# ============================================================
# Buildings trap heat. Trees cool areas. Let's see the relationships!

# Aggregate by census area
area_summary = df_with_heat.groupby('DA_ID').agg({
    'TREEID': 'count',
    'CANOPY_AREA_M2': 'sum',
    'LST_CELSIUS': 'first',
    'CANOPY_PCT': 'first',
    'NDBI': 'first',
    'HEAT_VULNERABILITY': 'first',
    'VULNERABILITY_CATEGORY': 'first',
    'POPULATION': 'first',
    'AREA_KM2': 'first'
}).reset_index()

area_summary.columns = ['DA_ID', 'Tree_Count', 'Total_Canopy_M2',
                        'Land_Surface_Temp', 'Existing_Canopy_Pct',
                        'Building_Density', 'Heat_Vulnerability',
                        'Vulnerability_Category', 'Population', 'Area_KM2']

# Calculate trees per capita and per km²
area_summary['Trees_Per_1000_People'] = (area_summary['Tree_Count'] / area_summary['Population'] * 1000).round(1)
area_summary['Trees_Per_KM2'] = (area_summary['Tree_Count'] / area_summary['Area_KM2']).round(1)

# Calculate correlations
print("📊 CORRELATION ANALYSIS: The Building-Heat-Tree Triangle")
print("=" * 60)
print(f"\n🌡️ Temperature vs Tree Canopy:     r = {area_summary['Land_Surface_Temp'].corr(area_summary['Existing_Canopy_Pct']):.3f}")
print(f"🏗️ Temperature vs Building Density: r = {area_summary['Land_Surface_Temp'].corr(area_summary['Building_Density']):.3f}")
print(f"🌳 Building Density vs Canopy:      r = {area_summary['Building_Density'].corr(area_summary['Existing_Canopy_Pct']):.3f}")

print("\n💡 Interpretation:")
print("   - Negative temp/canopy correlation = more trees → cooler areas")
print("   - Positive temp/building correlation = more concrete → hotter areas")
print("   - Negative building/canopy = buildings crowd out trees")

In [ ]:
# ============================================================
# INSIGHT 3: Finding Priority Planting Areas
# ============================================================
# The sweet spot: High heat vulnerability + Low tree coverage + Moderate building density
# (Very high building density means no space for trees!)

# Score each area for planting priority
# High score = good candidate for tree planting

area_summary['Planting_Priority'] = (
    area_summary['Heat_Vulnerability'] * 0.4 +  # Want high heat vulnerability
    (100 - area_summary['Existing_Canopy_Pct']) * 0.4 +  # Want LOW existing canopy
    (50 - abs(area_summary['Building_Density'] * 100)) * 0.2  # Moderate building density (space exists)
)

# Top priority areas
priority_areas = area_summary.nlargest(10, 'Planting_Priority')

print("🎯 TOP 10 PRIORITY AREAS FOR TREE PLANTING")
print("=" * 80)
print("Areas with: High heat need + Low existing canopy + Space available\n")
print(priority_areas[['DA_ID', 'Vulnerability_Category', 'Land_Surface_Temp',
                       'Existing_Canopy_Pct', 'Building_Density', 'Trees_Per_KM2',
                       'Planting_Priority']].to_string(index=False))

print(f"\n📍 Found {len(priority_areas)} high-priority areas for tree planting!")
print("   These neighborhoods would benefit most from increased tree coverage.")

### 🎓 What We Just Did: The Power of Data Integration

Each dataset told us something useful on its own. But **combining them** unlocks new insights:

| Single Dataset Insight | Combined Insight |
|------------------------|------------------|
| **Trees:** "Maples are 35% of our canopy - disease risk!" | + Heat data → "Maples are concentrated in already-cool areas" |
| **Heat:** "Some areas reach 40°C surface temperature" | + Trees → "Hot areas have 50% less canopy coverage" |
| **Buildings:** "Downtown has highest density (NDBI > 0.5)" | + Trees + Heat → "But 15% of hot areas have planting space!" |

**The Question We Can Now Answer:**
> "Where should we plant trees to help the people who need it most?"

**The Answer:** Priority areas where:
- Heat vulnerability is high (people are suffering)
- Existing canopy is low (trees are needed)
- Building density is moderate (space exists for planting)

**Real-World Impact:** This analysis could help Halifax allocate urban forestry budgets to the neighborhoods that need it most!

---

---
## 🎯 Your Turn: Data Combination Practice

Practice combining and analyzing integrated datasets!

| Level | Challenge | Skills Practiced |
|-------|-----------|------------------|
| **1** | Count how many trees are in 'Critical' vulnerability areas | Filtering, `value_counts()` |
| **2** | Calculate average trees per 1000 people for each vulnerability category | `groupby()`, derived metrics |
| **3** | Create your own priority scoring formula and justify your weights | Feature engineering, analysis |

In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 1 (Beginner)
# ============================================================
# How many trees are located in 'Critical' vulnerability areas?
# How does this compare to 'Low' vulnerability areas?

# YOUR CODE HERE:


In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 2 (Intermediate)
# ============================================================
# Calculate the average number of trees per 1000 people
# for each vulnerability category
#
# Which category has the MOST trees per capita?
# Which has the LEAST?

# YOUR CODE HERE:


In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 3 (Advanced)
# ============================================================
# Create your OWN priority scoring formula!
#
# The example used:
#   Heat_Vulnerability * 0.4 + (100 - Canopy) * 0.4 + (50 - Building) * 0.2
#
# Design your own formula and justify WHY you chose those weights.
# Consider:
#   - Should population density matter? (more people = more benefit)
#   - Should we penalize areas with NO space (very high building density)?
#   - What about existing tree age? (areas with old trees need replacements)
#
# Compare your top 10 priority areas to the original!

# YOUR CODE HERE:


## 4.5 Validate & Save 💾

In [ ]:
# Validation checks
print("🔍 VALIDATION CHECKS")
print("=" * 50)

# 1. Row count tracking
print(f"\n1. Row counts:")
print(f"   Original: {len(df)}")
print(f"   Cleaned: {len(df_clean)}")
print(f"   Lost: {len(df) - len(df_clean)} ({(len(df)-len(df_clean))/len(df)*100:.1f}%)")

# 2. Sanity checks
print(f"\n2. Sanity checks:")
assert df_clean['DBH'].min() >= 0, "Negative DBH found!"
print(f"   ✅ No negative DBH values")
assert df_clean['DBH'].max() <= 200, "DBH outliers remain!"
print(f"   ✅ No DBH outliers > 200cm")
assert df_clean['INSTYR'].isnull().sum() == 0, "Missing years remain!"
print(f"   ✅ No missing installation years")

# 3. Cross-validate with known facts
print(f"\n3. Reality check:")
maple_pct = df_clean['SP_COMM'].str.contains('Maple', case=False, na=False).mean() * 100
print(f"   Maple percentage: {maple_pct:.1f}% (expected: ~35-40%)")

print("\n✅ All validation checks passed!")

In [ ]:
# ============================================================
# 💾 Save your cleaned data to your workshop folder
# ============================================================

try:
    save_path = f'{workshop_folder}/halifax_trees_clean.csv'
    df_clean.to_csv(save_path, index=False)
    print(f"✅ Saved to: {save_path}")
    print(f"\n📊 Saved {len(df_clean)} clean tree records")
    print(f"📋 Columns: {list(df_clean.columns)}")
except Exception as e:
    print(f"❌ Error saving: {e}")
    print("   Make sure Google Drive is mounted and workshop_folder is set correctly.")

---
## 🎯 Your Turn: Validation Practice

Build robust validation into your data pipelines!

| Level | Challenge | Skills Practiced |
|-------|-----------|------------------|
| **1** | Add a validation check for valid coordinate ranges (Halifax is ~44.6°N, ~63.6°W) | Range checking, assertions |
| **2** | Create a before/after comparison showing how each cleaning step changed the data | Summary statistics, comparison |
| **3** | Write a reusable validation function that checks any tree DataFrame for common issues | Functions, comprehensive testing |

In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 1 (Beginner)
# ============================================================
# Add a validation check for valid coordinates
# Halifax is approximately at:
#   - Latitude: 44.6 to 44.8 (North)
#   - Longitude: -63.8 to -63.4 (West, so negative)
#
# Check that all trees fall within these bounds

# YOUR CODE HERE:


In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 2 (Intermediate)
# ============================================================
# Create a before/after comparison report showing:
#   - How many rows we started with vs ended with
#   - How DBH statistics changed (min, max, mean)
#   - How many unique species before vs after standardization
#   - How many missing values were filled

# YOUR CODE HERE:


In [ ]:
# ============================================================
# 🎯 YOUR TURN: Level 3 (Advanced)
# ============================================================
# Create a reusable validation function that checks a tree DataFrame
# and returns a report of any issues found
#
# The function should check:
#   1. No duplicate TREEIDs
#   2. No DBH values > 200 or < 0
#   3. No missing values in key columns (TREEID, SP_COMM, DBH)
#   4. Coordinates within Halifax bounds
#   5. INSTYR between 1800 and current year
#
# Return a dictionary with pass/fail status for each check

# YOUR CODE HERE:


---
# 🎉 Workshop Complete!

## What You Learned Today:

### Data Collection
- ✅ Open Data Portals (Halifax ArcGIS API)
- ✅ APIs (iNaturalist) and handling nested JSON
- ✅ Web scraping with BeautifulSoup
- ✅ The Inspect workflow for finding HTML structure

### Data Wrangling
- ✅ Inspecting data (`head()`, `info()`, `describe()`, `value_counts()`)
- ✅ Diagnosing issues (missing data, outliers, inconsistencies)
- ✅ Cleaning data (drop, fill, clip, standardize)
- ✅ Feature engineering (age, life stage, canopy area)
- ✅ Validation checks

## Key Takeaways:

1. **Always check for APIs first** before scraping
2. **73% of data science is data prep** - now you know why!
3. **Understand before you clean** - not all missing data is bad
4. **Document your decisions** - future you will thank present you
5. **Validate before you analyze** - garbage in = garbage out

---

## 📚 Resources

- **Halifax Open Data:** [data-hrm.hub.arcgis.com](https://data-hrm.hub.arcgis.com/)
- **iNaturalist API:** [api.inaturalist.org/v1/docs](https://api.inaturalist.org/v1/docs/)
- **pandas docs:** [pandas.pydata.org/docs](https://pandas.pydata.org/docs/)
- **BeautifulSoup docs:** [beautiful-soup-4.readthedocs.io](https://beautiful-soup-4.readthedocs.io/)

---

## 🏆 Stretch Challenges

1. **Get more trees** - modify the API query to get 5,000 or all 80,000 trees
2. **Search iNaturalist for invasive species** - try `taxon_name: "Acer platanoides"` (Norway Maple is invasive!)
3. **Create a "tree diversity index"** for different areas of Halifax
4. **Join with heat data** - load `heat_vulnerability_da.csv` and explore correlations

---
# Appendix: Quick Reference 📋

## pandas Cheat Sheet

```python
# INSPECTION
df.shape                    # (rows, columns)
df.dtypes                   # Column types
df.info()                   # Types + non-null counts
df.describe()               # Statistics
df['col'].value_counts()    # Frequency
df.isnull().sum()           # Missing count

# CLEANING
df.copy()                   # Make a copy first!
df.dropna()                 # Drop missing
df.fillna(value)            # Fill missing
df.drop_duplicates()        # Remove dupes
df['col'].str.strip()       # Remove whitespace
df['col'].clip(upper=100)   # Cap values
pd.to_numeric(df['col'], errors='coerce')

# TRANSFORMING
df['new'] = df['col'].apply(func)  # Apply function
df.groupby('col').agg({})   # Aggregate
df.merge(df2, on='key')     # Join
```

## BeautifulSoup Cheat Sheet

```python
soup = BeautifulSoup(html, 'html.parser')
soup.find('tag')                    # First match
soup.find_all('tag')                # All matches (list)
soup.find('div', class_='name')     # By class
element.get_text(strip=True)        # Extract text
element['href']                     # Get attribute
element.find_next('p')              # Next sibling of type
```